# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We will walk through loading, overview, extraction, EDA, and visualization using **record sets, fields, and columns referenced by their `@id`s** as defined by the Croissant standard.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset (metadata and croissant structure)
dataset = mlc.Dataset(croissant_url)

# Preview metadata (using the top-level properties)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Number of keywords: {len(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All are referenced by their `@id` field.

**Note:** The croissant specification uses the `recordSet` field as a list containing record set objects or `@id`s, each of which contains fields and columns. Let's enumerate them.

In [ ]:
# Collect the available record sets and preview their @id and field @id's.
recordsets = dataset.record_sets
if not recordsets:
    print('No record sets found! Attempting to list datatable resources directly (possible fallback)...')

for rs in recordsets:
    print(f"Record Set: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print('  Fields:')
        for fld in rs.fields:
            print(f"    - {fld.id} (name: {getattr(fld, 'name', None)})")
    if hasattr(rs, 'columns') and rs.columns:
        print('  Columns:')
        for col in rs.columns:
            print(f"    - {col.id} (name: {getattr(col, 'name', None)})")
    print()

# For reference, let's list all record set @id's as a list for the next step
all_recordset_ids = [rs.id for rs in recordsets]
print('Record Set @ids:', all_recordset_ids)
# Optionally print field ids for the first record set
if recordsets:
    field_ids = [fld.id for fld in getattr(recordsets[0], 'fields', [])]
    print('Field @ids in first record set:', field_ids)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s obtained from the overview step. For demonstration, we'll load all available record sets.

In [ ]:
# Extract all record sets to pandas DataFrames referenced by their @id
dataframes = {}

for record_set_id in all_recordset_ids:
    # Use mlcroissant's records() method, specifying the record set by its @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}.")

# Show the columns for the first loaded record set
if all_recordset_ids:
    main_record_set_id = all_recordset_ids[0]
    print(f"Columns for {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using Croissant `@id` fields for columns.

We'll perform:
- Filtering on a numeric field (e.g., age or interval)
- Normalization
- Grouping by a categorical field (e.g., sex or anatomical location)

> **Modify the `numeric_field_id` and `group_field_id` in the cell below to the relevant `@id` for your analysis. See the record set and field lists above for available options.

In [ ]:
import numpy as np

# Example: choose a record set and its numeric/categorical fields by @id
record_set_id = main_record_set_id  # The primary record set loaded earlier
df = dataframes[record_set_id]

# Example: suppose age field's @id is 'https://api.app.sen.science/frontiers/7862866/age',
# and sex's @id is 'https://api.app.sen.science/frontiers/7862866/sex'. Please replace with the real @id's from the overview step above.
# For demonstration, let's just use column names if @id's not available.

# Find a likely numeric field
numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or np.issubdtype(df[col].dtype, np.number)]
print(f"Numeric candidates: {numeric_candidates}")
numeric_field_id = numeric_candidates[0] if numeric_candidates else None

# Use a sample threshold value
threshold = 10

if numeric_field_id:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize this field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Pick a categorical/grouping field
    cat_candidates = [col for col in df.columns if df[col].dtype == object]
    if cat_candidates:
        group_field_id = cat_candidates[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using the loaded DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: histogram for a selected numeric field
if numeric_field_id is not None and not df[numeric_field_id].isnull().all():
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If a categorical group field is identified, show box plot
    if cat_candidates:
        group_field_id = cat_candidates[0]
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, extract, and perform basic EDA and visualization on a tabular Croissant dataset using only `@id` for schema elements. 

### Key points
- All dataset entities (record sets, fields, columns) are referenced by `@id`, ensuring unambiguous access following the Croissant specification.
- Data was loaded using `mlcroissant` and extracted to pandas DataFrames for further analysis.
- Numeric variables were identified, filtered, normalized, grouped, and visualized using standard Python data science tools.

**Next steps**: Adapt fields, code, and visualizations above for your specific analytical or scientific questions, or automate processing across multiple Croissant datasets.